# Notebook 3 · Who is lending well? (Question 3)

**Question:** How does credit performance compare across SBLCs, and has growth come at the cost of credit quality?

**Approach**
1. **Define a bad loan** carefully and test how much the definition matters.
2. **Show why timing matters:** charge-offs arrive years after approval, so young loans look artificially good.
3. **Compare fairly:** compare each SBLC with other lenders' loans of the *same approval year and loan size*.
4. **Check robustness:** a richer comparison, and a view without the biggest outlier.
5. **Link to growth:** did the fast growers lend worse?
6. **Explain why:** where do SBLC losses actually come from?

**Inputs:** `loans_clean.parquet` (Notebook 0), `sblc_list.csv` (Notebook 1), `sblc_growth_windows.csv` (Notebook 2)

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

In [2]:
df = pd.read_parquet("../../data/processed/loans_clean.parquet")
sblc = pd.read_csv("../../assignment1/outputs/sblc_list.csv")
gr = pd.read_csv("../../assignment1/outputs/sblc_growth_windows.csv", index_col=0)   # from Notebook 2

SHORT = {
    "Newtek Small Business Finance, Inc.": "Newtek", "VelocitySBA, LLC": "Velocity",
    "Readycap Lending, LLC": "Readycap", "Hana Small Business Lending, Inc.": "Hana",
    "Harvest Small Business Finance, LLC": "Harvest", "First Western SBLC, Inc": "First Western",
    "CRF Small Business Loan Company, LLC": "CRF", "Centerstone SBA Lending, Inc.": "Centerstone",
    "Grow America Fund, Incorporated": "Grow America", "Lendistry SBLC, LLC": "Lendistry",
    "Loan Source Incorporated": "Loan Source", "CapitalSpring SBLC, LLC": "CapitalSpring",
}
assert set(SHORT) == set(sblc["bankname"]), "SBLC list changed: update SHORT"
df["is_sblc"] = df["bankname"].isin(SHORT)
df["lender"] = df["bankname"].map(SHORT)
print("Loans:", len(df), "| SBLC loans:", df["is_sblc"].sum())

Loans: 478650 | SBLC loans: 11465


## Step 1 · What is a "bad loan"?
A **charge-off** is the lender writing the unpaid balance off as a loss. It is the clearest, most final sign of a bad loan in this data.

We define four versions:
- **Status = CHGOFF only:** the simple version. It undercounts, because some loans were charged off and later changed status (for example to "paid in full" after recoveries).
- **Ever charged off (our main definition):** the charge-off amount is above zero. This captures every loan that was ever written off. *Found in Notebook 0.*
- **Charged off or distress:** adds loans in liquidation, purchased by SBA, or closed after default.
- **Charged off within 60 months:** a fixed window, so old and young loans are judged over the same time.

In [3]:
df["bad"] = df["charged_off_ever"]                                   # main definition: charge-off amount > 0
df["bad_status"] = (df["loanstatus"] == "CHGOFF").astype(int)        # status only (sensitivity)
df["distress"] = df["loanstatus"].isin(["CHGOFF", "LIQUID", "PURCH(NOT C/O)", "CLSLN"]).astype(int)
df["bad_or_distress"] = ((df["bad"] == 1) | (df["distress"] == 1)).astype(int)

months_to_co = (df["chargeoffdate"] - df["approvaldate"]).dt.days / 30.4
df["bad60"] = ((df["bad"] == 1) & (months_to_co <= 60)).astype(int)  # charged off within 5 years

# Loan-size bands and product groups, used for fair comparisons
df["size_band"] = pd.cut(df["grossapproval"], [0, 50e3, 150e3, 350e3, 1e6, 5.1e6],
                         labels=["<=50k", "50-150k", "150-350k", "350k-1M", ">1M"])

def product_group(m):
    if m in ("Preferred Lenders Program", "Preferred Lenders with EWCP"): return "PLP"
    if "Express" in m: return "Express"
    if m == "Community Advantage Initiative": return "Community Advantage"
    return "Other"
df["product"] = df["processingmethod"].map(product_group)

### Does the definition change the answer?
If SBLCs look worse than other lenders under **every** definition, the conclusion is robust.

In [4]:
defs = (df.groupby("is_sblc")[["bad_status", "bad", "bad_or_distress", "bad60"]].mean() * 100).round(2)
defs.index = ["Other lenders (banks etc.)", "SBLCs"]
defs.columns = ["Status = CHGOFF only", "Ever charged off (main)", "Charged off or distress", "Charged off within 60 months"]
display(defs)

fig = px.bar(defs.reset_index().melt(id_vars="index"), x="variable", y="value", color="index", barmode="group",
             labels={"value": "% of loans", "variable": "", "index": ""},
             title="Bad-loan rate under four definitions: SBLCs vs other lenders")
fig.update_layout(template="plotly_white", height=420)
fig.show()

,Status = CHGOFF only,Ever charged off (main),Charged off or distress,Charged off within 60 months
Other lenders (banks etc.),6.33,7.59,9.06,4.27
SBLCs,11.06,15.51,20.09,9.40


## Step 2 · Timing: charge-offs arrive late
A loan approved in 2019 has had less time to fail than one approved in 2011. If we ignore this, recent vintages look better than they are. We measure how long charge-offs take.

In [5]:
co = df[df["bad"] == 1].copy()
co["months"] = (co["chargeoffdate"] - co["approvaldate"]).dt.days / 30.4
print(f"Median time from approval to charge-off: {co['months'].median():.0f} months")
for k in (36, 60, 84):
    print(f"Charge-offs that happen by {k} months: {(co['months'] <= k).mean():.0%}")

fig = px.ecdf(co.dropna(subset=["months"]), x="months", labels={"months": "Months from approval to charge-off"},
              title="How long do charge-offs take to appear?")
fig.update_layout(template="plotly_white", height=380); fig.update_yaxes(title="Share of all charge-offs")
fig.show()

display((df.groupby("approvalfy")[["bad", "bad60"]].mean() * 100).round(2).T)

Median time from approval to charge-off: 55 months
Charge-offs that happen by 36 months: 24%
Charge-offs that happen by 60 months: 56%
Charge-offs that happen by 84 months: 82%


approvalfy,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
bad,10.59,7.86,7.10,6.65,7.19,7.65,7.90,8.14,8.14,6.59
bad60,5.58,4.16,3.77,3.44,4.02,4.42,4.41,4.73,5.00,4.15


## Step 3 · The fair comparison
Comparing raw rates is unfair: lenders differ in **when** they lent (vintage) and **how much** they lent (loan size). Small loans fail more often, and older loans have had longer to fail.

**Method (observed vs expected):**
1. Group all loans by approval year and loan-size band.
2. In each group, compute the charge-off rate of **other lenders** (non-SBLCs). That is the *expected* rate.
3. For each SBLC, compare its actual rate (*observed*) with the expected rate for its own mix of loans.

**Observed / expected = 1.0** means the same as other lenders for comparable loans. **2.0** means twice as many charge-offs. We also show a 95% confidence interval, because a lender with 15 loans cannot be ranked like one with 5,000.

In [6]:
others = df[~df["is_sblc"]]

# Expected rate: other lenders, same approval year and loan-size band
exp1 = (others.groupby(["approvalfy", "size_band"], observed=True)["bad"].mean().rename("exp1"))
df = df.join(exp1, on=["approvalfy", "size_band"])
print("SBLC loans without a benchmark cell:", df.loc[df["is_sblc"], "exp1"].isna().sum())

# Richer expected rate (robustness): also matches on product type (PLP / Express / ...)
exp2 = (others.groupby(["approvalfy", "size_band", "product"], observed=True)["bad"].mean().rename("exp2"))
df = df.join(exp2, on=["approvalfy", "size_band", "product"])
df["exp2"] = df["exp2"].fillna(df["exp1"])

SBLC loans without a benchmark cell: 0


## Step 4 · SBLC scorecard
One row per lender: number of loans, charged-off loans, rates, observed/expected with its confidence interval, dollar losses, and **severity** (how much of a loan is lost when it is charged off).

In [7]:
def wilson(p, n, z=1.96):
    """95% confidence interval for a proportion (Wilson method, good for small samples)."""
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return centre - half, centre + half

s = df[df["is_sblc"]]
g = s.groupby("lender")
cr = g.agg(loans=("bad", "size"), gross_usd_m=("grossapproval", lambda x: x.sum() / 1e6),
           n_bad=("bad", "sum"), bad_rate=("bad", "mean"), status_rate=("bad_status", "mean"),
           bad60_rate=("bad60", "mean"), exp1=("exp1", "mean"), exp2=("exp2", "mean"),
           co_amt=("grosschargeoffamount", "sum"), gross=("grossapproval", "sum"))
cr["ci_lo"], cr["ci_hi"] = wilson(cr["bad_rate"], cr["loans"])
cr["obs_exp"] = cr["bad_rate"] / cr["exp1"]
cr["obs_exp_lo"] = cr["ci_lo"] / cr["exp1"]
cr["obs_exp_hi"] = cr["ci_hi"] / cr["exp1"]
cr["obs_exp_rich"] = cr["bad_rate"] / cr["exp2"]
cr["loss_rate_$"] = cr["co_amt"] / cr["gross"]                       # dollars charged off / dollars approved
cr["severity"] = s[s["bad"] == 1].groupby("lender").apply(
    lambda x: (x["grosschargeoffamount"] / x["grossapproval"]).mean(), include_groups=False)
cr = cr.sort_values("obs_exp")
display(cr[["loans", "n_bad", "bad_rate", "bad60_rate", "exp1", "obs_exp", "obs_exp_lo", "obs_exp_hi",
            "loss_rate_$", "severity"]].round(3))

fig = go.Figure(go.Scatter(
    x=cr["obs_exp"], y=cr.index, mode="markers",
    error_x=dict(type="data", symmetric=False, array=cr["obs_exp_hi"] - cr["obs_exp"],
                 arrayminus=cr["obs_exp"] - cr["obs_exp_lo"]),
    marker=dict(size=np.sqrt(cr["loans"]) / 2 + 6, color=np.where(cr["obs_exp"] > 1, "#e76f51", "#2a9d8f")),
    hovertemplate="%{y}: %{x:.2f}x<extra></extra>"))
fig.add_vline(x=1, line_dash="dash", annotation_text="same as other lenders")
fig.update_layout(title="Charge-offs vs expected (1.0 = same as other lenders for comparable loans)",
                  xaxis_title="Observed / expected charge-off rate (95% interval)", template="plotly_white", height=480)
fig.show()

,loans,n_bad,bad_rate,bad60_rate,exp1,obs_exp,obs_exp_lo,obs_exp_hi,loss_rate_$,severity
lender,,,,,,,,,,
Centerstone,205,0,0.000,0.000,0.038,0.000,0.000,0.482,0.000,NaN
Harvest,478,4,0.008,0.000,0.036,0.232,0.090,0.591,0.007,0.593
First Western,295,6,0.020,0.000,0.045,0.456,0.210,0.979,0.008,0.587
Loan Source,42,2,0.048,0.000,0.051,0.932,0.258,3.091,0.013,0.932
Hana,671,38,0.057,0.024,0.058,0.972,0.713,1.318,0.023,0.622
Grow America,268,17,0.063,0.004,0.063,1.001,0.631,1.566,0.050,0.554
Readycap,975,62,0.064,0.024,0.051,1.255,0.985,1.592,0.028,0.581
CRF,494,43,0.087,0.024,0.062,1.409,1.057,1.865,0.045,0.682
Newtek,2874,247,0.086,0.030,0.058,1.486,1.319,1.673,0.042,0.668


## Step 5 · Robustness: is one lender driving the result?
Velocity makes thousands of tiny, mostly unsecured Express loans. We test how much it drives the SBLC average, and whether a richer benchmark (also matched on product) changes the ranking.

In [8]:
pool_obs = s["bad"].mean(); pool_exp = s["exp1"].mean()
ex_v = s[s["lender"] != "Velocity"]
print(f"All SBLCs        : ever-charged-off {pool_obs:.1%} | other lenders {others['bad'].mean():.1%} | "
      f"observed/expected {pool_obs / pool_exp:.2f}x")
print(f"SBLCs ex-Velocity: ever-charged-off {ex_v['bad'].mean():.1%} | observed/expected {ex_v['bad'].mean() / ex_v['exp1'].mean():.2f}x")
print(f"Dollar loss rate (charge-off $ / approved $): SBLCs {s['grosschargeoffamount'].sum() / s['grossapproval'].sum():.1%}"
      f" | ex-Velocity {ex_v['grosschargeoffamount'].sum() / ex_v['grossapproval'].sum():.1%}"
      f" | other lenders {others['grosschargeoffamount'].sum() / others['grossapproval'].sum():.1%}")
print(f"Velocity share of all SBLC charged-off loans: {cr.loc['Velocity', 'n_bad'] / cr['n_bad'].sum():.0%}")
display(cr[["obs_exp", "obs_exp_rich"]].round(2).T)

All SBLCs        : ever-charged-off 15.5% | other lenders 7.6% | observed/expected 2.12x
SBLCs ex-Velocity: ever-charged-off 6.9% | observed/expected 1.26x
Dollar loss rate (charge-off $ / approved $): SBLCs 3.3% | ex-Velocity 3.1% | other lenders 2.9%
Velocity share of all SBLC charged-off loans: 75%


lender,Centerstone,Harvest,First Western,Loan Source,Hana,Grow America,Readycap,CRF,Newtek,Lendistry,CapitalSpring,Velocity
obs_exp,0.0,0.23,0.46,0.93,0.97,1.00,1.26,1.41,1.49,2.43,2.59,2.75
obs_exp_rich,0.0,0.22,0.50,1.04,1.02,0.96,1.23,1.18,1.34,1.26,2.28,2.76


## Step 6 · Did growth come at the cost of credit quality?
Two views:
1. **Scatter:** market-share change (from Notebook 2) against observed/expected charge-offs. Lenders in the **bottom right** gained share *and* lent well. The **top right** gained share but with weaker credit.
2. **Early vs late:** the same ratio for FY2010-14 and FY2015-19 loans, to see whether credit got better or worse as a lender scaled up.

Caution: later vintages are younger, so they have fewer charge-offs for everyone. The observed/expected ratio corrects for this because the benchmark is by approval year. Small lenders still have few events, so read them loosely.

In [9]:
credit_vs_growth = cr[["obs_exp", "loans"]].join(gr[["share_change_bps", "pool_share_late_%"]])
fig = px.scatter(credit_vs_growth.reset_index(), x="share_change_bps", y="obs_exp", size="loans", text="lender",
                 labels={"share_change_bps": "Change in market share (bps), FY2010-12 to FY2017-19",
                         "obs_exp": "Observed / expected charge-offs (1 = average)"},
                 title="Growth vs credit quality", size_max=45)
fig.add_hline(y=1, line_dash="dash"); fig.add_vline(x=0, line_dash="dash")
fig.update_traces(textposition="top center"); fig.update_layout(template="plotly_white", height=520)
fig.show()

s2 = s.assign(period=np.where(s["approvalfy"] <= 2014, "FY2010-14", "FY2015-19"))
lv = s2.groupby(["lender", "period"]).agg(loans=("bad", "size"), bad=("bad", "mean"), exp=("exp1", "mean"))
lv["obs_exp"] = lv["bad"] / lv["exp"]
half = lv["obs_exp"].unstack().join(lv["loans"].unstack(), rsuffix="_loans")
display(half.round(2))

/var/folders/15/81k2tcpd0pgc_fkw6m7t0kk00000gn/T/ipykernel_39683/316123491.py:13: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  half = lv["obs_exp"].unstack().join(lv["loans"].unstack(), rsuffix="_loans")


period,FY2010-14,FY2015-19,FY2010-14_loans,FY2015-19_loans
lender,,,,
CRF,1.44,1.40,167.0,327.0
CapitalSpring,3.32,0.00,12.0,3.0
Centerstone,NaN,0.00,NaN,205.0
First Western,0.47,0.43,146.0,149.0
Grow America,0.97,1.03,132.0,136.0
Hana,1.07,0.78,438.0,233.0
Harvest,0.00,0.27,44.0,434.0
Lendistry,NaN,2.43,NaN,140.0
Loan Source,1.14,0.00,31.0,11.0


## Step 7 · Why do lenders differ?
We compare SBLC and other-lender charge-off rates inside the same product, size, collateral and business-age groups. If SBLCs look the same as other lenders in one group and very different in another, that tells us where the gap comes from.

In [10]:
df["no_collateral"] = df["collateralind"].astype(str) != "True"
df["startup"] = df["businessage"].astype(str).str.contains("Startup|New", case=False)
s = df[df["is_sblc"]]            # refresh: s must include the two new columns

rows = []
for col in ["product", "size_band", "no_collateral", "startup"]:
    t = df.groupby([col, "is_sblc"], observed=True)["bad"].mean().unstack() * 100
    t.columns = ["Other lenders %", "SBLCs %"]
    t.index = [f"{col}: {i}" for i in t.index]
    rows.append(t)
drivers = pd.concat(rows).round(1)
display(drivers)

fig = px.bar(drivers.reset_index().melt(id_vars="index"), x="value", y="index", color="variable", barmode="group",
             orientation="h", labels={"value": "% ever charged off", "index": "", "variable": ""},
             title="Where do SBLCs differ from other lenders?")
fig.update_layout(template="plotly_white", height=650); fig.show()

# Lender-level view: what does each SBLC's loan book look like?
mix = s.groupby("lender").agg(median_loan=("grossapproval", "median"), avg_term=("terminmonths", "mean"),
                              startup=("startup", "mean"), no_collateral=("no_collateral", "mean"))
mix = mix.join(cr["obs_exp"])
display(mix.round(2))
print("Rank correlation with observed/expected (12 lenders, descriptive only):")
print(mix.corr(method="spearman")["obs_exp"].round(2).to_dict())

,Other lenders %,SBLCs %
product: Community Advantage,14.0,18.8
product: Express,8.2,27.4
product: Other,7.9,10.0
product: PLP,6.3,6.3
size_band: <=50k,9.6,26.4
size_band: 50-150k,8.6,8.9
size_band: 150-350k,6.8,8.6
size_band: 350k-1M,5.0,6.9
size_band: >1M,3.7,4.7
no_collateral: False,6.5,6.8


,median_loan,avg_term,startup,no_collateral,obs_exp
lender,,,,,
CRF,300000.0,138.94,0.48,0.01,1.41
CapitalSpring,483000.0,130.47,0.33,0.33,2.59
Centerstone,850000.0,194.01,0.09,0.00,0.00
First Western,1364000.0,290.54,0.05,0.01,0.46
Grow America,315000.0,169.93,0.03,0.01,1.00
Hana,528000.0,196.34,0.15,0.01,0.97
Harvest,927300.0,291.75,0.04,0.01,0.23
Lendistry,219000.0,106.48,0.21,0.00,2.43
Loan Source,800000.0,238.40,0.31,0.12,0.93


Rank correlation with observed/expected (12 lenders, descriptive only):
{'median_loan': -0.85, 'avg_term': -0.84, 'startup': 0.54, 'no_collateral': 0.46, 'obs_exp': 1.0}


## Save the scorecard

In [11]:
out = Path("../../assignment1/outputs"); out.mkdir(parents=True, exist_ok=True)
cr.to_csv(out / "sblc_credit_scorecard.csv")
half.to_csv(out / "sblc_credit_by_half_decade.csv")
drivers.to_csv(out / "credit_drivers_sblc_vs_others.csv")
print("saved")

saved


## Summary: who is lending well?

**1. SBLCs look twice as risky, but one lender drives it.**
SBLCs charged off 15.5% of loans against 7.6% for other lenders (worse under every definition). But Velocity, with tiny unsecured Express loans, accounts for about 75% of all SBLC charge-offs. **Without Velocity, SBLCs charge off 6.9% of loans, slightly better than other lenders.** Measured in dollars, SBLC losses are close to other lenders' (about 3.3% vs 2.9% of approved dollars).

**2. On the core PLP business, SBLCs perform exactly like banks** (6.3% vs 6.3%). The gap sits in small, unsecured Express loans (27% vs 8%) and loans under $50k (26% vs 10%).

**3. Fair ranking (observed / expected, 1.0 = average for comparable loans)**
- **Best:** Centerstone (no charge-offs in 205 loans), Harvest (0.23), First Western (0.46). These are large-loan, collateralised lenders. Centerstone and Harvest are also recent, so their record is short.
- **Around average:** Hana (0.97), Grow America (1.00), Readycap (1.26).
- **Worse:** Newtek (1.49, interval 1.32-1.67), CRF (1.41), Lendistry (2.43), Velocity (2.75).
- Loan Source and CapitalSpring are too small to judge.

**4. Growth and credit**
- **Newtek** is the biggest grower and it has *not* bought growth with worse credit: its ratio **improved from 2.1 to 1.3** as volume grew about 3.4x. It is still above average.
- **Readycap deteriorated**, from 0.82 to 1.64, while its volume nearly doubled.
- **Harvest** grew fast with very few charge-offs, but its recent loans are young, so a 0.27 ratio rests on very few events.
- **Hana** improved (1.07 to 0.78) but shrank.

**5. Why they differ:** loan size and collateral. Across the 12 lenders, bigger loans and longer terms go with fewer charge-offs (rank correlations about -0.85 and -0.84). This is only descriptive with 12 lenders. It is not proof of cause.

**Caveats**
- Recent vintages (FY2017-19) are still maturing, so their charge-off rates will rise somewhat.
- Several lenders have very few charge-offs, so their ratios have wide intervals.
- The benchmark is "other lenders" (mostly banks), whose product mix differs from SBLCs. The richer benchmark gives the same ranking, except that Lendistry drops from 2.43 to 1.26 because Community Advantage loans are riskier everywhere.